In [ ]:
# ============================================================================
# ETAPA 4 — BASELINE MLP COM SPLIT ESPACIAL
# ============================================================================
# Objetivo: Re-treinar o baseline (single-head MLP) usando o mesmo
# split espacial por hexágono que será usado no Multi-Head (etapa 5).
# Isso garante que a comparação baseline vs Multi-Head seja válida:
# mesmos dados, mesmo split, única variável = arquitetura.
#
# Diferenças em relação ao etapa3:
#   1. Modelo: BaselineMLP (single-head, sem gate)
#   2. Dados:  spatial_split_{train/val/test}.csv (split por hexágono)
#   3. Freeze: baseline_spatial_frozen/
#
# Tudo o mais é idêntico ao etapa3 (dataset, features, hiperparâmetros).
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import json
from datetime import datetime
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"✅ Configuração OK!")
print(f"   Device: {DEVICE}")
print(f"   PyTorch: {torch.__version__}")
print(f"   Notebook: ETAPA 4 — Baseline MLP + Split Espacial")

In [ ]:
# ============================================================================
# BASELINE MLP — ARQUITETURA
# ============================================================================
# Single-head MLP com a mesma capacidade total do Multi-Head:
#   Encoder:    287 → 256 → 128   (igual ao Multi-Head)
#   Head final: 128 → 64  → 1    (igual a uma head do Multi-Head)
#
# Nota: forward() aceita (x, spatial_features) para compatibilidade
# com o loop de treino do etapa3. spatial_features é ignorado.
# ============================================================================

class BaselineMLP(nn.Module):
    """Single-head MLP. Baseline para comparação com o Multi-Head."""

    def __init__(self, input_dim=287, hidden_dim=256,
                 encoder_dim=128, head_hidden_dim=64, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            # Encoder
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Head
            nn.Linear(encoder_dim, head_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x, spatial_features):
        # spatial_features ignorado — mantido apenas para compatibilidade
        # com o loop de treino (que passa spatial para o Multi-Head)
        prediction = self.net(x)
        gate_weights = None   # sem gate no baseline
        return prediction, gate_weights


# Verificação
model = BaselineMLP().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Teste de forward pass
dummy_x       = torch.randn(4, 287).to(DEVICE)
dummy_spatial = torch.zeros(4, 3).to(DEVICE)
out, gw = model(dummy_x, dummy_spatial)

print(f"✅ BaselineMLP criado!")
print(f"   Parâmetros: {total_params:,}")
print(f"   Output shape: {out.shape}  (esperado: [4, 1])")
print(f"   Gate weights: {gw}  (esperado: None)")
print()
print(f"   Arquitetura:")
print(f"     287 → 256 → 128 → 64 → 1 (sigmoid)")
print(f"     Dropout: 0.3 em todas as camadas")

In [ ]:
# ============================================================================
# DATASET — idêntico ao etapa3
# ============================================================================
# Copia exata do LULCMultiHeadDataset do etapa3.
# Não modificado — garante que as features extraídas são as mesmas.
# ============================================================================

import rasterio
from rasterio.windows import Window

DATA_DIR      = r"D:\Projetos\Cerrado\LULC"
PATTERN       = "brazil_coverage_{ano}_Cerrado.tif"
YEARS         = list(range(1985, 2025))
NODATA_IN     = 255
NODATA_OUT    = 0
PATCH_N       = 7
MAX_SERIE_LEN = len(YEARS) - 1   # 39
PATCH_YEARS   = 5


def _path(year, data_dir=DATA_DIR, pattern=PATTERN):
    return os.path.join(data_dir, pattern.format(ano=year))

def _ler_pixel(year, row, col, data_dir=DATA_DIR, pattern=PATTERN):
    with rasterio.open(_path(year, data_dir, pattern)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def _ler_patch(year, row, col, n, data_dir=DATA_DIR, pattern=PATTERN):
    half = n // 2
    with rasterio.open(_path(year, data_dir, pattern)) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr  = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)


class LULCMultiHeadDataset(Dataset):
    """
    Dataset com features de tamanho fixo (287).
    Idêntico ao etapa3 — não modificado.
    """

    def __init__(self, csv_path, indices=None,
                 data_dir=DATA_DIR, pattern=PATTERN,
                 years=YEARS, patch_n=PATCH_N,
                 max_serie_len=MAX_SERIE_LEN,
                 patch_years=PATCH_YEARS):

        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['label'].notna()].reset_index(drop=True)

        if indices is not None:
            self.df = self.df.iloc[indices].reset_index(drop=True)

        self.data_dir     = data_dir
        self.pattern      = pattern
        self.years        = years
        self.patch_n      = patch_n
        self.max_serie_len = max_serie_len
        self.patch_years  = patch_years

        print(f"  Dataset: {len(self.df):,} samples")
        print(f"  Labels: {self.df['label'].value_counts().to_dict()}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r   = self.df.iloc[idx]
        row = int(r["row"])
        col = int(r["col"])
        T   = int(r["T"])
        label = int(r["label"])

        # 1. Série temporal (sempre 39)
        anos_serie = [y for y in self.years if y < T]
        if len(anos_serie) > 0:
            serie_raw = np.array(
                [_ler_pixel(y, row, col, self.data_dir, self.pattern)
                 for y in anos_serie], dtype=np.float32)
            serie_raw = np.where(serie_raw == NODATA_IN,
                                 NODATA_OUT, serie_raw).astype(np.float32)
        else:
            serie_raw = np.array([], dtype=np.float32)

        serie_len = len(serie_raw)
        serie     = np.zeros(self.max_serie_len, dtype=np.float32)
        if serie_len > 0:
            serie[self.max_serie_len - serie_len:] = serie_raw

        # 2. Patch espacial (sempre 5×7×7 = 245)
        anos_patch = [y for y in self.years if y < T][-self.patch_years:]
        patch = np.zeros((self.patch_years, self.patch_n, self.patch_n),
                         dtype=np.float32)
        for i, y in enumerate(anos_patch):
            patch_idx = self.patch_years - len(anos_patch) + i
            patch[patch_idx] = _ler_patch(y, row, col, self.patch_n,
                                          self.data_dir, self.pattern)

        # 3. Features auxiliares (sempre 3)
        if serie_len > 0:
            has_21    = float(np.sum(serie_raw == 21)) / serie_len
            anos_past = 0
            for v in reversed(serie_raw):
                if v == 15: anos_past += 1
                else:       break
            cl_tm1 = float(serie_raw[-1])
        else:
            has_21 = anos_past = cl_tm1 = 0.0

        aux_features = np.array([has_21, float(anos_past), cl_tm1],
                                dtype=np.float32)

        # 4. Concatenar → [287]
        patch_flat = patch.flatten()
        features   = np.concatenate([serie, patch_flat, aux_features])
        assert features.shape == (287,), \
            f"Features shape errado: {features.shape} (idx={idx}, T={T})"

        # 5. Spatial features para gate [3]
        # No baseline, o gate não existe — mas mantemos o retorno para
        # compatibilidade com o loop de treino.
        if 'pattern_code' in self.df.columns and pd.notna(r.get('pattern_code')):
            pattern = str(r['pattern_code'])
            is_cluster_conv = 1.0 if pattern.startswith('CC') else 0.0
            is_cluster_use  = 1.0 if 'C' in pattern[-2:] else 0.0
            consol_level    = 2.0 if pattern.startswith('CC') else 1.0
        else:
            is_cluster_conv = is_cluster_use = 0.5
            consol_level    = 1.5

        spatial = np.array([is_cluster_conv, is_cluster_use, consol_level],
                           dtype=np.float32)

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(spatial,  dtype=torch.float32),
            torch.tensor([label],  dtype=torch.float32)
        )


print("✅ LULCMultiHeadDataset definido (idêntico ao etapa3)")

In [ ]:
# ============================================================================
# CARREGAR DADOS — SPLIT ESPACIAL POR HEXÁGONO
# ============================================================================
# Diferença central em relação ao etapa3:
#   Etapa3: random_split de um CSV único (72% overlap espacial)
#   Etapa4: três CSVs pré-divididos por hexágono (0% overlap)
# ============================================================================

BASE_DIR  = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
SPLIT_DIR = BASE_DIR / "spatial_split"

print("Carregando splits espaciais...")
print("⚠️  Vai extrair features REAIS dos rasters (~1-2 min por epoch)")
print()

print("Train:")
train_dataset = LULCMultiHeadDataset(SPLIT_DIR / "spatial_split_train.csv")
print()
print("Val:")
val_dataset   = LULCMultiHeadDataset(SPLIT_DIR / "spatial_split_val.csv")
print()
print("Test:")
test_dataset  = LULCMultiHeadDataset(SPLIT_DIR / "spatial_split_test.csv")

print(f"\n✅ Splits carregados:")
print(f"   Train: {len(train_dataset):,}")
print(f"   Val:   {len(val_dataset):,}")
print(f"   Test:  {len(test_dataset):,}")
print(f"   Split: ESPACIAL por hexágono (0% overlap garantido)")

BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f"\n✅ DataLoaders prontos! Batch size: {BATCH_SIZE}")

# Sanity check
print("\n🔍 Testando extração de features...")
sample_features, sample_spatial, sample_label = train_dataset[0]
assert sample_features.shape == (287,), f"Erro: {sample_features.shape}"
print(f"   Features shape: {sample_features.shape}  ✅")
print(f"   Spatial shape:  {sample_spatial.shape}   ✅")
print(f"   Label:          {sample_label.item()}")

In [ ]:
# ============================================================================
# FUNÇÕES DE TREINO — idênticas ao etapa3
# ============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    """Treina uma epoch."""
    model.train()
    total_loss = correct = total = 0

    pbar = tqdm(loader, desc="Training")
    for features, spatial, labels in pbar:
        features = features.to(device)
        spatial  = spatial.to(device)
        labels   = labels.to(device)

        predictions, _ = model(features, spatial)
        loss = criterion(predictions, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred_class  = (predictions > 0.5).float()
        correct    += (pred_class == labels).sum().item()
        total      += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'acc':  f'{100*correct/total:.1f}%'})

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Avalia modelo."""
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for features, spatial, labels in tqdm(loader, desc="Evaluating"):
            features = features.to(device)
            spatial  = spatial.to(device)
            labels   = labels.to(device)

            predictions, _ = model(features, spatial)
            loss = criterion(predictions, labels)

            total_loss += loss.item()
            pred_class  = (predictions > 0.5).float()
            correct    += (pred_class == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    preds_np      = np.array(all_preds).flatten()
    labels_np     = np.array(all_labels).flatten()
    pred_class_np = (preds_np > 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_np, pred_class_np, average='binary')
    try:
        auc = roc_auc_score(labels_np, preds_np)
    except:
        auc = 0.0

    return {
        'loss':      total_loss / len(loader),
        'accuracy':  correct / total,
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'auc':       auc
    }


print("✅ Funções de treino definidas!")

In [ ]:
# ============================================================================
# TREINO — hiperparâmetros idênticos ao etapa3
# ============================================================================

# Reinicializar modelo (garante treino do zero)
torch.manual_seed(SEED)
model = BaselineMLP().to(DEVICE)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True)

N_EPOCHS        = 30
EARLY_STOP_PAT  = 10

best_val_loss    = float('inf')
best_model_state = None
best_val_metrics = None
patience_counter = 0
history          = {'train_loss': [], 'train_acc': [], 'val_metrics': []}

print("=" * 70)
print("INICIANDO TREINO — BASELINE MLP (SPLIT ESPACIAL)")
print("=" * 70)
print(f"  Modelo:     BaselineMLP ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Split:      Espacial por hexágono (0% overlap)")
print(f"  Train:      {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")
print(f"  Epochs:     {N_EPOCHS} (early stopping patience={EARLY_STOP_PAT})")
print(f"  LR:         0.001 (ReduceLROnPlateau patience=5)")
print(f"  Device:     {DEVICE}")
print()

for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print("-" * 70)

    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, DEVICE)
    val_metrics = evaluate(
        model, val_loader, criterion, DEVICE)

    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_metrics'].append(val_metrics)

    print(f"\nTrain - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f}, "
          f"Acc: {val_metrics['accuracy']:.4f}, "
          f"F1: {val_metrics['f1']:.4f}, "
          f"AUC: {val_metrics['auc']:.4f}")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss    = val_metrics['loss']
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        best_val_metrics = val_metrics.copy()
        patience_counter = 0
        print("  → Best model updated!")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PAT:
            print(f"\nEarly stopping após {epoch+1} epochs.")
            break

    print()

print("=" * 70)
print("TREINO CONCLUÍDO")
print("=" * 70)
print(f"  Epochs rodadas: {len(history['train_loss'])}")
print(f"  Melhor val loss: {best_val_loss:.4f}")
print(f"  Melhor val acc:  {best_val_metrics['accuracy']:.4f}")
print(f"  Melhor val F1:   {best_val_metrics['f1']:.4f}")

In [ ]:
# ============================================================================
# AVALIAÇÃO FINAL NO TEST SET
# ============================================================================

# Carregar melhor modelo
model.load_state_dict(best_model_state)
model.to(DEVICE)

print("=" * 70)
print("AVALIAÇÃO FINAL — TEST SET (SPLIT ESPACIAL)")
print("=" * 70)

test_metrics = evaluate(model, test_loader, criterion, DEVICE)

val_acc  = best_val_metrics['accuracy']
test_acc = test_metrics['accuracy']
val_test_gap = test_acc - val_acc

print(f"\n{'Métrica':<15} {'Validation':>12} {'Test':>12}")
print("-" * 42)
for k in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    print(f"{k:<15} {best_val_metrics[k]:>12.4f} {test_metrics[k]:>12.4f}")

print(f"\nVal-Test Gap: {val_test_gap*100:+.2f}pp")
print(f"  (positivo = overfitting; negativo = test > val)")

print("\n" + "=" * 70)
print("RESUMO PARA O PAPER")
print("=" * 70)
print(f"  Test Accuracy:  {test_acc:.1%}")
print(f"  Test F1:        {test_metrics['f1']:.3f}")
print(f"  Test AUC:       {test_metrics['auc']:.3f}")
print(f"  Val-Test Gap:   {val_test_gap*100:+.1f}pp")
print(f"  Split:          Espacial por hexágono")

In [ ]:
# ============================================================================
# CONGELAR MODELO BASELINE — VERSÃO FINAL
# ============================================================================

BASE_DIR   = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
FREEZE_DIR = BASE_DIR / "baseline_spatial_frozen"
FREEZE_DIR.mkdir(exist_ok=True, parents=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 70)
print("CONGELANDO BASELINE MLP")
print("=" * 70)
print(f"  Timestamp:  {timestamp}")
print(f"  Diretório:  {FREEZE_DIR}")

# 1. Checkpoint .pth
checkpoint = {
    'model_state_dict': best_model_state,
    'model_class':      'BaselineMLP',
    'config': {
        'input_dim':      287,
        'hidden_dim':     256,
        'encoder_dim':    128,
        'head_hidden_dim': 64,
        'dropout':        0.3
    },
    'metrics': {
        'validation': {k: float(v) for k, v in best_val_metrics.items()},
        'test':       {k: float(v) for k, v in test_metrics.items()},
        'val_test_gap': float(val_test_gap)
    },
    'training': {
        'n_epochs':   len(history['train_loss']),
        'batch_size': BATCH_SIZE,
        'lr':         0.001,
        'optimizer':  'Adam',
        'criterion':  'BCELoss',
        'seed':       SEED
    },
    'dataset': {
        'split_method':   'hexagon_stratified_by_Class8590',
        'split_metadata': str(SPLIT_DIR / 'spatial_split_metadata.json'),
        'train_samples':  len(train_dataset),
        'val_samples':    len(val_dataset),
        'test_samples':   len(test_dataset),
        'spatial_overlap_pct': 0
    }
}

pth_path = FREEZE_DIR / f"baseline_spatial_frozen_{timestamp}.pth"
torch.save(checkpoint, pth_path)
size_mb = pth_path.stat().st_size / (1024**2)
print(f"\n✅ Modelo salvo: {pth_path.name} ({size_mb:.2f} MB)")

# 2. JSON legível
results = {
    'model_info': {
        'name':        'Baseline MLP',
        'version':     'v1.0-spatial-split',
        'frozen_date': timestamp,
        'architecture': '287→256→128→64→1 (sigmoid)'
    },
    'performance': {
        'validation': {k: str(round(float(v), 4))
                       for k, v in best_val_metrics.items()},
        'test':       {k: str(round(float(v), 4))
                       for k, v in test_metrics.items()},
        'generalization': {
            'val_test_gap':    str(round(val_test_gap, 4)),
            'gap_percentage':  f"{val_test_gap*100:.2f}pp",
            'status':          'EXCELLENT' if abs(val_test_gap) < 0.05
                               else 'ACCEPTABLE' if abs(val_test_gap) < 0.10
                               else 'OVERFITTING'
        }
    },
    'dataset_info': {
        'source_file':   'treino_balanceado_FINAL.csv',
        'split_method':  'hexagon_stratified_by_Class8590',
        'spatial_overlap_train_test': '0%',
        'splits': {
            'train': len(train_dataset),
            'val':   len(val_dataset),
            'test':  len(test_dataset)
        }
    },
    'training': {
        'epochs_run':   len(history['train_loss']),
        'early_stopped': len(history['train_loss']) < N_EPOCHS,
        'best_val_loss': round(best_val_loss, 4)
    }
}

json_path = FREEZE_DIR / f"results_{timestamp}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✅ Resultados salvos: {json_path.name}")

# 3. Resumo final
print("\n" + "=" * 70)
print("BASELINE CONGELADO COM SUCESSO")
print("=" * 70)
print(f"\n📁 {FREEZE_DIR}")
print(f"\n📊 Performance (split espacial):")
print(f"  Val Accuracy:   {val_acc:.1%}")
print(f"  Test Accuracy:  {test_acc:.1%}")
print(f"  Test F1:        {test_metrics['f1']:.3f}")
print(f"  Test AUC:       {test_metrics['auc']:.3f}")
print(f"  Val-Test Gap:   {val_test_gap*100:+.1f}pp")
print()
print(f"🎯 Próximo passo: etapa5 — Multi-Head com o mesmo split")
print(f"   Usar os mesmos CSVs em SPLIT_DIR = {SPLIT_DIR}")